# MediSense-Cardio -- Methodology, Explained

**How to use this notebook:** read each grey markdown box out loud, then run the code cell right
below it (click the play button, or Shift+Enter). The real output appears directly under that
cell. Go top to bottom, in order. Everything here is real, working code -- nothing is
pre-recorded.


## Part 1: the overall research methodology (30 seconds)

We take ECG recordings where the correct answer is already known. We turn each one into a
printed-looking page, damage the page on purpose in six different ways (blur, tilt, JPEG
compression, low resolution, faded grid, uneven lighting) at five levels of severity, run every
damaged page through the reading app, and compare its answer to the truth we already know.
**50 recordings x 31 versions = 1,550 tests**, done twice.

The rest of this notebook zooms into one specific piece of that pipeline: **how do we take a
photo and automatically tell how tilted it is?** This is part of the next stage of the project
(automatically detecting a bad photo), and it is the clearest single piece to demonstrate live,
because it is fast, needs no GPU, and can be tested with a number picked on the spot.


## Step 1: mount Drive and load one real study page


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
STUDY_DIR = '/content/drive/MyDrive/MediSense-Cardio-Study'
CLEAN_DIR = f'{STUDY_DIR}/images_clean'
clean_files = sorted(glob.glob(f'{CLEAN_DIR}/*.png'))
print(f'{len(clean_files)} clean study images found.')
assert len(clean_files) > 0, 'No clean images found -- check Drive access.'
IMG_PATH = clean_files[0]
print('Using a REAL page from the study:', IMG_PATH)


## Step 2: convert the picture to grayscale

**What this does:** a photo is normally stored as three numbers per pixel (red, green, blue).
We don't need color to find edges or lines, so we collapse it down to ONE number per pixel:
how bright or dark it is, from 0 (black) to 255 (white). This makes every later step simpler
and faster, without losing anything we actually need.

**Formula:** each pixel's brightness is a weighted average of its red, green and blue values
(this weighting matches how the human eye perceives brightness -- green looks brighter to us
than blue at the same intensity):

$$brightness = 0.299 \times R + 0.587 \times G + 0.114 \times B$$


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread(IMG_PATH)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

print(f'Original: {img.shape[1]} x {img.shape[0]} pixels, 3 numbers per pixel (color)')
print(f'Grayscale: {gray.shape[1]} x {gray.shape[0]} pixels, 1 number per pixel (brightness)')
print(f'Brightness ranges from {int(gray.min())} (darkest) to {int(gray.max())} (brightest)')

fig, axs = plt.subplots(1, 2, figsize=(14, 6))
axs[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axs[0].set_title('Original (color)'); axs[0].axis('off')
axs[1].imshow(gray, cmap='gray'); axs[1].set_title('Grayscale (brightness only)'); axs[1].axis('off')
plt.tight_layout(); plt.show()


## Step 3: find the edges

**What this does:** we check every pixel and ask, "is this noticeably brighter or darker than
the pixels right next to it?" If yes, we mark it white -- that's an edge. If the area around it
is smooth and unchanging, we mark it black -- nothing there. This is a standard, well-known
technique called **Canny edge detection**. The result is a black-and-white sketch of every
boundary in the picture: the grid lines, the heartbeat trace, the text.

**Formula:** not one formula, three steps:
1. Measure how fast brightness changes in the x direction ($G_x$) and y direction ($G_y$) at
   every pixel, using a small 3x3 comparison window (the Sobel operator).
2. Combine them into one "how strong is the edge here" number:
   $$strength = \sqrt{G_x^2 + G_y^2}$$
3. Keep only pixels above a threshold (we used 50-150) that are the strongest point along their
   own edge direction.


In [ ]:
edges = cv2.Canny(gray, 50, 150)
n_edge_px = int((edges > 0).sum())
total_px = edges.size
print(f'{n_edge_px:,} of {total_px:,} pixels were marked as an edge ({100*n_edge_px/total_px:.2f}%)')
print('This is a real count of actual pixels, not an estimate.')

fig, axs = plt.subplots(1, 2, figsize=(14, 6))
axs[0].imshow(gray, cmap='gray'); axs[0].set_title('PREVIOUS STEP: grayscale'); axs[0].axis('off')
axs[1].imshow(edges, cmap='gray'); axs[1].set_title('THIS STEP: edges found (Canny)'); axs[1].axis('off')
plt.tight_layout(); plt.show()


## Step 4: find every straight line among those edges

**What this does:** the edge sketch is just scattered white dots -- it doesn't know what a
"line" is yet. We run a second standard technique, the **Hough transform**, which scans those
edge dots and reports back every straight run it can find.

**The one important fix we had to make:** if we accept lines of ANY length, the wiggly heartbeat
trace itself gets picked up as thousands of tiny fake "lines," pointing every which way, and
they drown out the real, straight lines from the page (the grid, the borders). So we add one
rule: **only trust a line if it's at least 30% as long as the picture's width.** The heartbeat
trace's pieces are all short and curved, so this rule throws them out automatically, leaving only
the page's genuine straight structure.

**Formula:** every possible straight line can be written as:
$$\rho = x \cos\theta + y \sin\theta$$
where $\rho$ is the line's distance from the corner of the image and $\theta$ is its angle. For
every edge pixel, we compute this for every possible $\theta$ and let each one "vote." A line is
a ($\rho$, $\theta$) pair that gets a lot of votes -- meaning a lot of edge pixels agree it's
part of the same straight line.


In [ ]:
min_len = int(0.3 * gray.shape[1])
print(f'Rule: only keep a line if it is at least {min_len} pixels long (30% of the {gray.shape[1]}px width)')

lines_any_length = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=100, minLineLength=1, maxLineGap=10)
lines_long_only = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=100, minLineLength=min_len, maxLineGap=10)
n_any = 0 if lines_any_length is None else len(lines_any_length)
n_long = 0 if lines_long_only is None else len(lines_long_only)
print(f'Without the rule: {n_any} lines found (mostly noise from the wiggly trace)')
print(f'With the rule:    {n_long} lines found (the real page structure)')

annotated = img.copy()
if lines_long_only is not None:
    lines_arr = lines_long_only.reshape(-1, 4) if lines_long_only.ndim == 3 else lines_long_only
    for x1, y1, x2, y2 in lines_arr:
        cv2.line(annotated, (x1, y1), (x2, y2), (255, 0, 0), 4)

fig, axs = plt.subplots(1, 2, figsize=(16, 8))
axs[0].imshow(edges, cmap='gray'); axs[0].set_title('PREVIOUS STEP: edges'); axs[0].axis('off')
axs[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axs[1].set_title(f'THIS STEP: {n_long} long lines found (drawn in red)'); axs[1].axis('off')
plt.tight_layout(); plt.show()


## Step 5: turn all those lines into one "how tilted is this page" number

**What this does:** each surviving line has its own angle. A well-aligned page has all its lines
sitting at 0 degrees (flat) or 90 degrees (upright). For every line, we measure how far it is
from the nearest one of those two. Then we take the **middle value (median)** across all the
lines -- that's our single answer.

**Formula, each line's angle:** using the line's two end points, converted to degrees:
$$angle = \arctan2(y_2 - y_1,\ x_2 - x_1)$$

**Formula, the final answer:** the median (middle value) of all the individual line angles --
not an average, so one wildly wrong line can't drag the answer off course.


In [ ]:
angles = []
if lines_long_only is not None:
    lines_arr = lines_long_only.reshape(-1, 4) if lines_long_only.ndim == 3 else lines_long_only
    for x1, y1, x2, y2 in lines_arr:
        a = np.degrees(np.arctan2(y2 - y1, x2 - x1)) % 180
        dev = a % 90
        angles.append(min(dev, 90 - dev))

print(f'All {len(angles)} individual angle measurements:')
print([round(a, 2) for a in angles])
measured = float(np.median(angles)) if angles else None
print(f'\nMedian (our final answer): {measured:.3f} degrees' if measured is not None else 'Not enough lines found.')
print(f'\nThis page is clean/undamaged, so this should be close to 0 -- and it is.')


## Step 6: the proof -- does this actually work on a page we DON'T already know the answer for?

**The test:** rotate this page by an angle you pick, out loud, right now. The code below
is never told that angle -- it only sees the rotated picture, and repeats Steps 2-5 above on it.
If the number it reports back matches what you typed in, that's the proof the method
genuinely works -- it isn't a memorized answer.

**Change the number below to whatever you say, then run this cell.** Run it again with a
different number for as many tests as they want.


In [ ]:
TEST_ANGLE = 7  # <-- change this to whatever you pick, then re-run this cell

def rotate_by_any_angle(image, angle_deg):
    h, w = image.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle_deg, 1.0)
    return cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=(255, 255, 255))

def measure_tilt(image):
    g = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    e = cv2.Canny(g, 50, 150)
    ml = int(0.3 * g.shape[1])
    ln = cv2.HoughLinesP(e, 1, np.pi/180, threshold=100, minLineLength=ml, maxLineGap=10)
    if ln is None or len(ln) < 3:
        return None, 0
    arr = ln.reshape(-1, 4) if ln.ndim == 3 else ln
    devs = []
    for x1, y1, x2, y2 in arr:
        a = np.degrees(np.arctan2(y2 - y1, x2 - x1)) % 180
        d = a % 90
        devs.append(min(d, 90 - d))
    return float(np.median(devs)), len(arr)

rotated_test = rotate_by_any_angle(img, float(TEST_ANGLE))
result, n_found = measure_tilt(rotated_test)

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(rotated_test, cv2.COLOR_BGR2RGB))
plt.title(f'Rotated by {TEST_ANGLE} degrees (the number you picked)'); plt.axis('off')
plt.show()

if result is None:
    print('Too few long lines survived at this exact angle -- a known, honest limitation.')
    print('Try a slightly different number.')
else:
    expected = abs(float(TEST_ANGLE)) % 90
    expected = min(expected, 90 - expected)
    print(f'Lines found: {n_found}')
    print(f'You typed in:      {TEST_ANGLE} degrees')
    print(f'Code measured:     {result:.3f} degrees')
    print(f'Difference:        {abs(expected - result):.3f} degrees')


## Summary, in one sentence

We turned a photo into a black-and-white edge sketch, found every straight line in that sketch
while filtering out the wiggly heartbeat trace, measured each line's angle, combined them into
one number, and just proved live that this number matches a rotation nobody told the code about.
